*[Cell 1]*

# Chapter 9: Streaming Medallion Architecture, Bronze layer

**Databricks Free Edition version: runs entirely from UC volumes, no Kafka.** The book streams from a Kafka (Redpanda) topic in Docker. Here the source is a **landing folder in a UC volume**:

| Book (Docker + Kafka) | Free Edition (UC volumes) |
|---|---|
| Kafka topic `ecomm.v1.clickstream` | landing folder `/Volumes/workspace/dldg/work/ch09/landing/ecomm.v1.clickstream/` |
| `rpk topic produce ...` | `produce()` appends a file of JSON events to the landing folder |
| `spark.readStream.format("kafka")` | `spark.readStream.format("text")` on the landing folder |
| Kafka offsets in the checkpoint | list of already-read files in the checkpoint |

The event data comes from the ecommerce CSV samples in `/Volumes/workspace/dldg/datasets/ecomm_behavior_data/csv/`. The bronze table, silver checkpoints, and landing files all live under `/Volumes/workspace/dldg/work/ch09/`.

Bronze keeps the raw record as bytes in `value`, plus `topic` and `timestamp` metadata, so the book's silver code works unchanged.

In [2]:
# [Cell 2]
# Databricks Free Edition: Spark runs on serverless via Databricks Connect (see dldg.py at the repo root)
from dldg import spark, DATASETS, WORK, tree, rm, batch_rows
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, to_date, lit, struct, to_json, date_format
from pyspark.sql.streaming import DataStreamReader, DataStreamWriter

from typing import Dict

CH = f"{WORK}/ch09"  # replaces /opt/spark/work-dir/data
TOPIC_NAME = "ecomm.v1.clickstream"
LANDING = f"{CH}/landing/{TOPIC_NAME}"  # UC volume landing folder (replaces the Kafka topic)

rm(CH)  # start clean: landing files, bronze table, and checkpoints

*[Cell 3]*

## Land records in the volume
Replaces `rpk topic produce ecomm.v1.clickstream`. Each call to `produce()` writes one new file to the landing folder, like an upstream system dropping a batch of events.

We produce:
1. The two sample records from the chapter README.
2. One malformed record, so the silver layer's `_corrupt` handling has something to catch.
3. The October sample of the [Kaggle ecommerce dataset](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store) (64 rows).

In [3]:
# [Cell 4]
def produce(df: DataFrame) -> None:
    """Append rows to the landing folder. df must have a single string column named `value`."""
    df.coalesce(1).write.mode("append").text(LANDING)

def ecomm_csv_as_json(file_name: str) -> DataFrame:
    """Read an ecommerce CSV sample and serialize each row to the JSON the book's producer sends."""
    csv = (spark.read.option("header", True).option("inferSchema", True)
           .csv(f"{DATASETS}/ecomm_behavior_data/csv/{file_name}"))
    return csv.select(to_json(struct(
        date_format("event_time", "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("event_time"),
        *[c for c in csv.columns if c != "event_time"])).alias("value"))

readme_samples = [
    '{"event_time": "2023-08-30T00:00:00Z","event_type": "view","product_id": 4782,"category_id": 2053013552326770905,"category_code": "appliances.environment.water_heater","brand": "heater3","price": 2789.0,"user_id": 195,"user_session": "19ae88e1-4a02-4b57-94a8-a46f6c6c60c4"}',
    '{"event_time": "2023-09-21T00:00:00Z","event_type": "view","product_id": 4783,"category_id": 2051113552326770905,"category_code": "appliances.televisions","brand": "sony","price": 2789.0,"user_id": 196,"user_session": "19ae88e1-4a02-4b57-94a8-a46f6c6c60c4"}',
    '{"event_time": "2023-09-21T00:00:00Z", "event_type": "view", "product_id": "not-a-number"',  # malformed
]
produce(spark.createDataFrame([(s,) for s in readme_samples], "value string"))
produce(ecomm_csv_as_json("2019-Oct-sm.csv"))
tree(f"{CH}/landing")

ecomm.v1.clickstream/
    _SUCCESS
    _committed_4644199513528900545
    _committed_8646635297271265018
    _started_4644199513528900545
    _started_8646635297271265018
    part-00000-tid-4644199513528900545-9c1d28da-af1a-4883-af79-2d12330d1cb0-242-1-c000.txt
    part-00000-tid-8646635297271265018-dd63bfb7-06e7-4a7b-be8f-8b77a5803ee8-245-1-c000.txt


*[Cell 5]*

## Source: the UC volume landing folder
In the book, the source is a Kafka broker in a Docker container:
~~~
kafka.brokers=PLAINTEXT://kafka-rp:29092
kafka.topic=ecomm.v1.clickstream
kafka.group.id=delta-dldg-1
~~~

Here, a streaming `text` source reads the landing folder. `to_bronze_columns()` gives it the same columns as the book's bronze table; `timestamp` is the file's modification time (when the batch landed).

In [14]:
# [Cell 6]
# example 11-3
# Book version (needs a Kafka broker, not available on Free Edition):
# reader_opts: Dict[str, str] = {
#     "subscribe": "ecomm.v1.clickstream",
#     "startingOffsets": "earliest",
#     "minPartitions": "4",
#     "kafka.bootstrap.servers": "PLAINTEXT://kafka-rp:29092",
#     "groupIdPrefix": "delta-dldg-1"
# }
# source_df: DataStreamReader = spark.readStream.options(**reader_opts).format("kafka")

reader_opts: Dict[str, str] = {
    "maxFilesPerTrigger": "1",  # like Kafka's maxOffsetsPerTrigger: limits each micro-batch
}
writer_opts: Dict[str, str] = {
    "checkpointLocation": f"{CH}/_checkpoints/bronze",  # serverless requires an explicit checkpoint
    "mergeSchema": "false",
    "overwriteSchema": "false",
    "path": f"{CH}/delta/bronze_raw/"
}

def to_bronze_columns(df: DataFrame) -> DataFrame:
    """Shape the landed records into the bronze columns the book uses (value as bytes, plus metadata)."""
    return df.select(
        lit(None).cast("binary").alias("key"),
        col("value").cast("binary").alias("value"),  # raw record bytes, parsed in silver
        lit(TOPIC_NAME).alias("topic"),
        col("_metadata.file_modification_time").alias("timestamp"),
    )

source_df: DataStreamReader = (
    spark.readStream
    .options(**reader_opts)
    .format("text")
)

# From here on, the book's code unchanged (plus .transform(to_bronze_columns))
bronze_layer_stream: DataStreamWriter = (
    source_df.load(LANDING)
    .transform(to_bronze_columns)
    .select(col("key"),col("value"),col("topic"),col("timestamp"))
    .withColumn("event_date", to_date(col("timestamp")))
    .writeStream
    .format('delta')
    .options(**writer_opts)
    .partitionBy("event_date")
    .trigger(availableNow=True)
)

In [15]:
# [Cell 7]
streaming_query = bronze_layer_stream.start()
streaming_query.awaitTermination()  # availableNow: processes every landed file, then stops

In [16]:
# [Cell 8]
streaming_query.status

{'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}

In [17]:
# [Cell 9]
# One entry per micro-batch: with maxFilesPerTrigger=1, one per file in the landing folder
batch_rows(streaming_query)

[(3, 0)]

In [18]:
# [Cell 10]
# Already stopped (availableNow), so this is a no-op; kept from the book
streaming_query.stop()

*[Cell 11]*

## Produce more, then rerun the same stream
With Kafka, a restarted stream resumes from the offsets in its checkpoint. The file source does the same with the list of files it has read. Produce the November sample (105 rows) and start the stream again: only the new file is processed.

In [10]:
# [Cell 12]
produce(ecomm_csv_as_json("2019-Nov-sm.csv"))

streaming_query = bronze_layer_stream.start()
streaming_query.awaitTermination()
batch_rows(streaming_query)

[(2, 105)]

In [11]:
# [Cell 13]
# The bronze table: raw bytes plus landing metadata, nothing parsed yet
bronze = spark.read.format("delta").load(writer_opts["path"])
print(bronze.count(), "rows")
bronze.selectExpr("CAST(value AS STRING) AS value", "topic", "timestamp", "event_date").show(5, truncate=80)

172 rows
+--------------------------------------------------------------------------------+--------------------+-------------------+----------+
|                                                                           value|               topic|          timestamp|event_date|
+--------------------------------------------------------------------------------+--------------------+-------------------+----------+
|{"event_time":"2019-11-01T00:00:00Z","event_type":"view","product_id":1003461...|ecomm.v1.clickstream|2026-11-01 21:55:08|2026-11-01|
|{"event_time":"2019-11-01T00:00:00Z","event_type":"view","product_id":5000088...|ecomm.v1.clickstream|2026-11-01 21:55:08|2026-11-01|
|{"event_time":"2019-11-01T00:00:01Z","event_type":"view","product_id":1730266...|ecomm.v1.clickstream|2026-11-01 21:55:08|2026-11-01|
|{"event_time":"2019-11-01T00:00:01Z","event_type":"view","product_id":3601530...|ecomm.v1.clickstream|2026-11-01 21:55:08|2026-11-01|
|{"event_time":"2019-11-01T00:00:01Z","event_t

In [12]:
# [Cell 14]
spark.sql(f"DESCRIBE HISTORY delta.`{writer_opts['path']}`").select("version", "operation", "operationParameters", "operationMetrics.numOutputRows").show(truncate=60)

+-------+----------------+------------------------------------------------------------+-------------+
|version|       operation|                                         operationParameters|numOutputRows|
+-------+----------------+------------------------------------------------------------+-------------+
|      3|STREAMING UPDATE|{outputMode -> Append, queryId -> a32ba385-9512-45b0-914a...|          105|
|      2|STREAMING UPDATE|{outputMode -> Append, queryId -> a32ba385-9512-45b0-914a...|           64|
|      1|STREAMING UPDATE|{outputMode -> Append, queryId -> a32ba385-9512-45b0-914a...|            3|
|      0|STREAMING UPDATE|{outputMode -> Append, queryId -> a32ba385-9512-45b0-914a...|         NULL|
+-------+----------------+------------------------------------------------------------+-------------+

